# Random Forest 
Works by creating multiple decision trees (in regression case: regression trees), where the output is the average of the predictions of the trees. Works for nonlinear relationships between features and target. Random forests correct for decision trees' tendency toward overfitting. 

**Justification:** Decision trees are invariant under scaling and transformations of feature values, robust to inclusion of irrelevant features, produce inspectable models, but are not highly accurate. Trees that are grown very deep (XX) can learn irregular patterns (overfitting, low bias, high variance). Random forests average deep decision trees, attempting to reduce variance, at the cost of some bias. 





#### `RandomForestRegressor`
Each tree in the ensemble is built from a sample drawn with replacement (i.e., a bootstrap sample) from the training set. During the construction of each tree in the forest, a random subset of the features is considered. The size of this subset is controlled by the `max_features` parameter; it may include either all input features or a random subset of them. 

#### Parameters
The main parameters to adjust when using these methods is `n_estimators` and `max_features`. 

The former is the number of trees in the forest. The larger the better, but also the longer it will take to compute. In addition, note that results will stop getting significantly better beyond a critical number of trees. 

The latter is the size of the random subsets of features to consider when splitting a node. The lower the greater the reduction of variance, but also the greater the increase in bias. Empirical good default values are `max_features=1.0` or equivalently `max_features=None` (always considering all features instead of a random subset) for regression problems. 

The default value of `max_features=1.0` is equivalent to bagged trees and more randomness can be achieved by setting smaller values (e.g. 0.3 is a typical default in the literature). 

Good results are often achieved when setting `max_depth=None` in combination with `min_samples_split=2` (i.e., when fully developing the trees). Bear in mind though that these values are usually not optimal, and might result in models that consume a lot of RAM. The best parameter values should always be cross-validated. 

In addition, note that in random forests, bootstrap samples are used by default (`bootstrap=True`) while the default strategy for extra-trees is to use the whole dataset (`bootstrap=False`). When using bootstrap sampling the generalization error can be estimated on the left out or out-of-bag samples. This can be enabled by setting `oob_score=True`.

#### `ExtraTreesRegressor`

As in random forests, a random subset of candidate features is used, but instead of looking for the most discriminative thresholds, thresholds are drawn at random for each candidate feature and the best of these randomly-generated thresholds is picked as the splitting rule. This usually allows to reduce the variance of the model a bit more, at the expense of a slightly greater increase in bias.

#### Geographic Handling
Latitude and longitude are fed in as features, the model does not inherently understand the correlation between nearby geographic points. 

Random Forests stuggle to extrapolate. Predictions in low sampled regions may be unreliable [XX]. 

# 1. Import and Load

In [1]:
import sys
from pathlib import Path

# Set root
cwd = Path.cwd()
ROOT = next(p for p in [cwd, *cwd.parents] if (p / ".git").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.features.preprocessing import load_model_data
from src.evaluation import splits
from src.evaluation.make_splits import load_split

from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier 
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error

In [2]:
# Load the cleaned/merged dataset

X, y, meta = load_model_data()
FEATURES = list(X.columns)

data = load_split(pd.concat([meta, X], axis=1).assign(log_nitrate=y))
print(data["fold"].value_counts().sort_index().to_dict(), "| holdout rows:", int(data["holdout"].sum()))

# df = pd.read_csv(DATA_PATH, dtype={"gm_well_id": "string"}, parse_dates=["gm_samp_collection_date"])
# df = load_split(df)

# print(df["fold"].value_counts().sort_index().to_dict(), "| holdout rows:", int(df["holdout"].sum()))

{0: 1035, 1: 972, 2: 1143, 3: 1041, 4: 1089} | holdout rows: 1089


In [3]:
# Set the training folds and the holdout

train_val = data[~data["holdout"]].reset_index(drop=True)       # Training set with 4 folds


X_tv = train_val[FEATURES]
y_tv = train_val["log_nitrate"]

# Print the sizes of the training folds
print(train_val["fold"].value_counts().sort_index().to_dict())

{0: 1035, 1: 972, 2: 1143, 3: 1041}


# 0. Dummy Model

In [4]:
def make_model():
    """Replace with your model. A new, untrained model is created for every fold."""
    return DummyRegressor(strategy="mean")        # baseline: always predicts the training mean

oof_pred = np.full(len(train_val), np.nan)        # out-of-fold prediction for every row
fold_scores = []

for train, test in splits.iter_folds(train_val["fold"]):
    model = make_model()
    model.fit(X_tv[train], y_tv[train])
    oof_pred[test] = model.predict(X_tv[test])

    fold = int(train_val.loc[test, "fold"].iloc[0])
    fold_scores.append({"fold": fold,
                        "rmse_log": root_mean_squared_error(y_tv[test], oof_pred[test])})

fold_scores = pd.DataFrame(fold_scores)
exceeds = np.expm1(y_tv) > 10
print(fold_scores.round(3).to_string(index=False))
print(f"Overall RMSE (log): {root_mean_squared_error(y_tv, oof_pred):.3f}")
print(f"Accuracy of the >10 mg/L call: {np.mean((np.expm1(oof_pred) > 10) == exceeds):.3f}")

 fold  rmse_log
    0     1.227
    1     1.183
    2     1.222
    3     1.314
Overall RMSE (log): 1.238
Accuracy of the >10 mg/L call: 0.662


# 1. RandomForestRegressor
We need to optimize the hyperparameters `n_estimators` and `max_features`

Because we have hyperparameters, we need to use nested CV. 

In [5]:
from src.models import random_forest
development, X_dev, y_dev = random_forest.make_inner_subfolds(data, FEATURES, N_SUBFOLDS = 4, THRESHOLD_MG_L = 10, SEED = 77)


subfold    0    1    2    3
fold                       
0        187  195  237  416
1        109  194  230  439
2        408  258  302  175
3        186  160  224  471


In [6]:
all_best_params = []

for seed in range(100):
    outer_results, best_params_by_fold = (
        random_forest.fit_RFR_with_nested_cv(
            development,
            X_dev,
            y_dev,
            N_SUBFOLDS=4,
            THRESHOLD_MG_L=10,
            SEED=seed,
        )
    )

    # Record the seed for each outer fold's best parameters
    params = best_params_by_fold.copy()
    params["seed"] = seed

    all_best_params.append(params)

all_best_params = pd.concat(all_best_params, ignore_index=True)

## This is going to take forever to run


In [7]:
all_best_params.to_csv("all_best_params.csv")

In [10]:
# Ranks the combinations by how often they were selected across all seeds and outer folds.
overall_frequency = (
    all_best_params
    .groupby(["max_features", "n_estimators"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

overall_frequency["frequency"] = (
    overall_frequency["count"] / len(all_best_params)
)

print(f"Overall frequency:")
print(f"{overall_frequency}")
#Find the most frequent combination for each outer fold
fold_frequency = (
    all_best_params
    .groupby(["outer_fold", "max_features", "n_estimators"])
    .size()
    .reset_index(name="count")
)

# Normalize within each outer fold
fold_frequency["frequency"] = (
    fold_frequency["count"]
    / fold_frequency.groupby("outer_fold")["count"].transform("sum")
)

fold_frequency = fold_frequency.sort_values(
    ["outer_fold", "count"],
    ascending=[True, False],
)

#print(f"The most frequent combination for each outer fold:")
#print(f"{fold_frequency.to_string(index=False)}")

most_frequent_per_fold = (
    fold_frequency
    .sort_values(
        ["outer_fold", "count", "n_estimators"],
        ascending=[True, False, True],
    )
    .drop_duplicates("outer_fold")
)

print(f"Extract only the most frequent combination for each outer fold:")
print(f"{most_frequent_per_fold.to_string(index=False)}")

Overall frequency:
  max_features  n_estimators  count  frequency
4         log2           500    160     0.4000
3         log2           300     75     0.1875
8         sqrt           500     68     0.1700
7         sqrt           300     33     0.0825
2         log2           100     24     0.0600
6         sqrt           100     20     0.0500
1         log2            50     11     0.0275
5         sqrt            50      8     0.0200
0          0.5           100      1     0.0025
Extract only the most frequent combination for each outer fold:
 outer_fold max_features  n_estimators  count  frequency
          0         log2           500     30       0.30
          1         log2           500     37       0.37
          2         log2           500     56       0.56
          3         log2           500     37       0.37


In [11]:
random_forest.fit_RFR_with_normal_cv(development, X_dev, y_dev, 500, "log2", THRESHOLD_MG_L = 10, SEED = 77)

CV results (outer fold RMSE):
 outer_fold  rmse_log
          0     1.039
          1     1.131
          2     1.085
          3     1.032
Overall RMSE (log): 1.072
Accuracy of the >10 mg/L call: 0.727


# RandomForestClassifier

In [ ]:
#r2_score
#scores['auc'].mean():.3f} | Mean Brier skill: {scores['brier_skill'].mean():.3f}"

# Final Validation

In [ ]:
# ============================================================
# FINAL EVALUATION ON THE HOLD-OUT FOLD
# Run this ONCE, for the final chosen model only, after all
# model choices (features, hyperparameters) are finished.
# ============================================================
RUN_FINAL_EVALUATION = False

if RUN_FINAL_EVALUATION:
    holdout_df = df[df["holdout"]].reset_index(drop=True)
    final_model = ...                      # the chosen model, refit on all of train_val
    final_model.fit(X, y)
    holdout_pred = final_model.predict(holdout_df[FEATURES])
    ...                                    # scores on np.log1p(holdout_df[TARGET])
